In [68]:
pip install sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 3.5 MB/s eta 0:00:006 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 3.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 4.9 MB/s eta 0:00:00 MB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 3.8 MB/s eta 0:00:004.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 3.4 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 3.9 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.2
    Uninstalling sympy-1.13.2:
      Successfully uninstalled sympy-1.13.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sentence_transformers
import faiss
import pandas
print("Libraries loaded successfully")

Libraries loaded successfully


In [5]:
import pandas as pd

df = pd.read_csv("kk_14.csv")

print(df.shape)
df.head()

(70613, 7)


,INSTITUTE NAME,INSTI_ADDRESS,STATE,WEBSITE,FIELD,PLACEMENT,ACCREDITED_BY
0,INDIAN INSTITUTE OF TECHNOLOGY MADRAS,"INDIAN INSTITUTE OF TECHNOLOGY MADRAS, KAMBAR ...",CHENNAI,https://en.wikipedia.org/wiki/IIT_Madras,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, NAAC ACCREDITATION"
1,INDIAN INSTITUTE OF TECHNOLOGY DELHI,"INDIAN INSTITUTE OF TECHNOLOGY DELHI, HAUZ KHA...",NEW DELHI,https://en.wikipedia.org/wiki/IIT_Delhi,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED"
2,INDIAN INSTITUTE OF TECHNOLOGY BOMBAY,"INDIAN INSTITUTE OF TECHNOLOGY BOMBAY, POWAI, ...",MUMBAI,https://en.wikipedia.org/wiki/IIT_Bombay,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED"
3,INDIAN INSTITUTE OF TECHNOLOGY KANPUR,"INDIAN INSTITUTE OF TECHNOLOGY KANPUR, MDR78C,...",KANPUR,https://en.wikipedia.org/wiki/IIT_Kanpur,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, NBA ACCREDITED"
4,INDIAN INSTITUTE OF TECHNOLOGY KHARAGPUR,"INDIAN INSTITUTE OF TECHNOLOGY KHARAGPUR, BOUN...",KHARAGPUR,https://www.iitkgp.ac.in/,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED"


In [11]:
#Create Searchable Text Column
df["SEARCH_TEXT"] = (
    df["FIELD"].fillna("") + " " +
    df["STATE"].fillna("") + " " +
    df["INSTITUTE NAME"].fillna("")
)

In [13]:
df[["FIELD","STATE","INSTITUTE NAME","SEARCH_TEXT"]].head()

,FIELD,STATE,INSTITUTE NAME,SEARCH_TEXT
0,ENGINEERING COLLEGE,CHENNAI,INDIAN INSTITUTE OF TECHNOLOGY MADRAS,ENGINEERING COLLEGE CHENNAI INDIAN INSTITUTE O...
1,ENGINEERING COLLEGE,NEW DELHI,INDIAN INSTITUTE OF TECHNOLOGY DELHI,ENGINEERING COLLEGE NEW DELHI INDIAN INSTITUTE...
2,ENGINEERING COLLEGE,MUMBAI,INDIAN INSTITUTE OF TECHNOLOGY BOMBAY,ENGINEERING COLLEGE MUMBAI INDIAN INSTITUTE OF...
3,ENGINEERING COLLEGE,KANPUR,INDIAN INSTITUTE OF TECHNOLOGY KANPUR,ENGINEERING COLLEGE KANPUR INDIAN INSTITUTE OF...
4,ENGINEERING COLLEGE,KHARAGPUR,INDIAN INSTITUTE OF TECHNOLOGY KHARAGPUR,ENGINEERING COLLEGE KHARAGPUR INDIAN INSTITUTE...


In [17]:
#Load Sentence Transformer Model
#Error displaying widget”

This happens because Jupyter doesn't have the progress-bar widget installed that sentence-transformers tries to use.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded


In [21]:
#GENERATE EMBEDDINGS
embeddings = model.encode(
    df["SEARCH_TEXT"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/2207 [00:00<?, ?it/s]

In [23]:
print(embeddings.shape)

(70613, 384)


In [25]:
#CREATE A FAISS VECTOR
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Index size:", index.ntotal)

Index size: 70613


In [31]:
#CREATE SEARCH FUNCTION
def search_colleges(query, k=5):
    
    query_vector = model.encode([query])
    
    distances, indices = index.search(query_vector, k)
    
    results = df.iloc[indices[0]][[
        "INSTITUTE NAME",
        "STATE",
        "FIELD",
        "PLACEMENT",
        "ACCREDITED_BY",
        "WEBSITE"
    ]]
    
    return results

In [33]:
#TESTING QUERIES
search_colleges("engineering colleges in delhi")

,INSTITUTE NAME,STATE,FIELD,PLACEMENT,ACCREDITED_BY,WEBSITE
27403,G.B. PANT GOVT. ENGINEERING COLLEGE,DELHI,ENGINEERING COLLEGE,AVERAGE,"AICTE APPROVED, UGC RECOGNIZED",NOT AVAILABLE
27381,B.K. INSTITUTE OF EDUCATION & TECHNOLOGY,DELHI,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, NAAC ACCREDITATION",NOT AVAILABLE
11287,DELHI ENGINEERING COLLEGE,HARYANA,ENGINEERING COLLEGE,AVERAGE,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE
1,INDIAN INSTITUTE OF TECHNOLOGY DELHI,NEW DELHI,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED",https://en.wikipedia.org/wiki/IIT_Delhi
11766,DELHI COLLEGE OF ENGINEERING,NAN,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, NBA ACCREDITED",https://en.wikipedia.org/wiki/Delhi_Technologi...


In [37]:
# AGAIN TESTING BLINDO
search_colleges("law colleges")

,INSTITUTE NAME,STATE,FIELD,PLACEMENT,ACCREDITED_BY,WEBSITE
20054,SAIN LAW COLLEGE,UTTAR PRADESH,LAW COLLEGE,GOOD,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE
35674,MORADABAD COLLEGE OF LAW,UTTAR PRADESH,LAW COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED",NOT AVAILABLE
37333,LMS LAW COLLEGE,MANIPUR,LAW COLLEGE,GOOD,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE
19898,KOUSHALENDRA RAO LAW COLLEGE,CHHATISGARH,LAW COLLEGE,GOOD,"AICTE APPROVED, NAAC ACCREDITATION",NOT AVAILABLE
30528,J.R.S.E.T COLLEGE OF LAW,WEST BENGAL,LAW COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED",https://en.wikipedia.org/wiki/J.R.S.E.T._Colle...


In [39]:
search_colleges("engineering colleges in karnataka")

,INSTITUTE NAME,STATE,FIELD,PLACEMENT,ACCREDITED_BY,WEBSITE
52810,"CITY ENGINEERING COLLEGE, BANGALORE",KARNATAKA,ENGINEERING COLLEGE,AVERAGE,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE
52841,"GOVT. ENGINEERING COLLEGE, RAMANAGARA",KARNATAKA,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, UGC RECOGNIZED",NOT AVAILABLE
70050,"CITY ENGINEERING COLLEGE, BANGALORE (ID: C-1307)",KARNATAKA,ENGINEERING COLLEGE,GOOD,"AICTE APPROVED, NBA ACCREDITED",https://cityengineeringcollege.ac.in/
1244,"PRESIDENCY COLLEGE, BANGALORE",KARNATAKA,ENGINEERING,AVERAGE,"AICTE APPROVED, UGC RECOGNIZED",https://www.presidencycollege.ac.in/
1237,"THE NATIONAL DEGREE COLLEGE, BANGALORE",KARNATAKA,ENGINEERING,AVERAGE,"AICTE APPROVED, UGC RECOGNIZED",https://collegedunia.com/college/28327-the-nat...


In [41]:
search_colleges("indian institute of technology")

,INSTITUTE NAME,STATE,FIELD,PLACEMENT,ACCREDITED_BY,WEBSITE
545,GANDHI INSTITUTE FOR TECHNOLOGY,ORISSA,ENGINEERING,GOOD,"AICTE APPROVED, NAAC ACCREDITATION",NOT AVAILABLE
11673,INDIAN INSTITUTE OF MANAGEMENT AHMEDABAD,AHMEDABAD,TECHNOLOGY COLLEGE,AVERAGE,"AICTE APPROVED, UGC RECOGNIZED",NOT AVAILABLE
28810,BALAJI INSTITUTE OF I.T & MANAGEMENT,ANDHRA PRADESH,TECHNOLOGY COLLEGE,AVERAGE,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE
575,INDIAN INSTITUTE OF MANAGEMENT,TAMIL NADU,"ENGINEERING, TECHNOLOGY COLLEGE",AVERAGE,"AICTE APPROVED, UGC RECOGNIZED",https://www.iimb.ac.in/
60960,INSTITUTE OF MANAGEMENT & TECHNOLGOY,HARYANA,TECHNOLOGY COLLEGE,AVERAGE,"AICTE APPROVED, NBA ACCREDITED",NOT AVAILABLE


In [43]:
#Save Model + Index
import pickle

faiss.write_index(index, "college_index.faiss")

with open("college_df.pkl", "wb") as f:
    pickle.dump(df, f)

print("Index saved")

Index saved
